# 1. Contexto y carga del dataset

En este notebook se realizará el perfilamiento del dataset de los premios GRAMMY a partir de la tabla `raw.grammy`, almacenada en PostgreSQL.

El objetivo de esta primera etapa es:

- Establecer conexión con PostgreSQL.
- Cargar los datos directamente desde la capa RAW.
- Confirmar el volumen de registros y columnas.
- Verificar que la estructura recuperada corresponde con la fuente extraída.

En esta etapa no se realizarán transformaciones ni limpieza de datos. Las modificaciones semánticas, normalizaciones y conversiones de tipos serán evaluadas posteriormente a partir de los hallazgos del profiling.

**Flujo:**

PostgreSQL → `raw.grammy` → DataFrame → perfilamiento

In [88]:
import pandas as pd
from sqlalchemy import create_engine


# ---------------------------------------------------------
# PostgreSQL connection
# ---------------------------------------------------------

DB_URL = (
    "postgresql+psycopg2://"
    "etl_user:etl_password@localhost:5433/etl_workshop"
)

engine = create_engine(DB_URL)


# ---------------------------------------------------------
# Load RAW dataset
# ---------------------------------------------------------

query = """
SELECT *
FROM raw.grammy;
"""

df = pd.read_sql(query, engine)


# ---------------------------------------------------------
# Initial verification
# ---------------------------------------------------------

print(f"Records loaded: {len(df)}")
print(f"Columns loaded: {len(df.columns)}")

df.head()

Records loaded: 4810
Columns loaded: 10


,year,title,published_at,updated_at,category,nominee,artist,workers,img,winner
0,2019,62nd Annual GRAMMY Awards (2019),19/05/2020 5:10,19/05/2020 5:10,Record Of The Year,Bad Guy,Billie Eilish,"Finneas O'Connell, producer; Rob Kinelski & Fi...",https://www.grammy.com/sites/com/files/styles/...,VERDADERO
1,2019,62nd Annual GRAMMY Awards (2019),19/05/2020 5:10,19/05/2020 5:10,Record Of The Year,"Hey, Ma",Bon Iver,"BJ Burton, Brad Cook, Chris Messina & Justin V...",https://www.grammy.com/sites/com/files/styles/...,VERDADERO
2,2019,62nd Annual GRAMMY Awards (2019),19/05/2020 5:10,19/05/2020 5:10,Record Of The Year,7 rings,Ariana Grande,"Charles Anderson, Tommy Brown, Michael Foster ...",https://www.grammy.com/sites/com/files/styles/...,VERDADERO
3,2019,62nd Annual GRAMMY Awards (2019),19/05/2020 5:10,19/05/2020 5:10,Record Of The Year,Hard Place,H.E.R.,"Rodney “Darkchild” Jerkins, producer; Joseph H...",https://www.grammy.com/sites/com/files/styles/...,VERDADERO
4,2019,62nd Annual GRAMMY Awards (2019),19/05/2020 5:10,19/05/2020 5:10,Record Of The Year,Talk,Khalid,"Disclosure & Denis Kosiak, producers; Ingmar C...",https://www.grammy.com/sites/com/files/styles/...,VERDADERO


# 2. Perfil estructural

En esta etapa se analizará la estructura técnica del dataset cargado desde `raw.grammy`.

El objetivo es identificar:

- Número de registros y columnas.
- Nombre de cada campo.
- Tipo de dato técnico utilizado por Pandas.
- Cantidad y porcentaje de valores no nulos.
- Cardinalidad de cada columna.
- Posible interpretación semántica inicial de los campos.

Este análisis permitirá diferenciar entre la representación técnica de los datos en RAW y su significado dentro del dominio de los premios GRAMMY.

No se realizarán conversiones ni transformaciones en esta etapa.

In [89]:
structure_profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum().values,
    "null": df.isna().sum().values,
    "null_percentage": (
        df.isna().mean().values * 100
    ).round(2),
    "unique": df.nunique(dropna=True).values
})

structure_profile

,column,dtype,non_null,null,null_percentage,unique
year,year,str,4810,0,0.00,62
title,title,str,4810,0,0.00,62
published_at,published_at,str,4810,0,0.00,4
updated_at,updated_at,str,4810,0,0.00,8
category,category,str,4810,0,0.00,638
nominee,nominee,str,4804,6,0.12,4131
artist,artist,str,2970,1840,38.25,1658
workers,workers,str,2620,2190,45.53,2366
img,img,str,3443,1367,28.42,1463
winner,winner,str,4810,0,0.00,1


In [90]:
semantic_dictionary = pd.DataFrame([
    ["year", "Año de la edición", "Temporal / categórica", "Año"],
    ["title", "Nombre de la edición", "Texto", "Texto"],
    ["published_at", "Fecha de publicación", "Temporal", "Fecha y hora"],
    ["updated_at", "Fecha de actualización", "Temporal", "Fecha y hora"],
    ["category", "Categoría del premio", "Categórica", "Categoría"],
    ["nominee", "Nominado", "Texto / entidad", "Texto"],
    ["artist", "Artista asociado", "Texto / entidad", "Texto"],
    ["workers", "Profesionales asociados", "Texto / multivaluado", "Texto"],
    ["img", "Imagen asociada", "Texto / URL", "URL"],
    ["winner", "Indicador de ganador", "Categórica binaria", "VERDADERO / FALSO"],
], columns=[
    "column",
    "meaning",
    "semantic_type",
    "scale_or_unit"
])

semantic_dictionary

,column,meaning,semantic_type,scale_or_unit
0,year,Año de la edición,Temporal / categórica,Año
1,title,Nombre de la edición,Texto,Texto
2,published_at,Fecha de publicación,Temporal,Fecha y hora
3,updated_at,Fecha de actualización,Temporal,Fecha y hora
4,category,Categoría del premio,Categórica,Categoría
5,nominee,Nominado,Texto / entidad,Texto
6,artist,Artista asociado,Texto / entidad,Texto
7,workers,Profesionales asociados,Texto / multivaluado,Texto
8,img,Imagen asociada,Texto / URL,URL
9,winner,Indicador de ganador,Categórica binaria,VERDADERO / FALSO


In [91]:
structure_profile = structure_profile.merge(
    semantic_dictionary,
    on="column",
    how="left"
)

structure_profile

,column,dtype,non_null,null,null_percentage,unique,meaning,semantic_type,scale_or_unit
0,year,str,4810,0,0.00,62,Año de la edición,Temporal / categórica,Año
1,title,str,4810,0,0.00,62,Nombre de la edición,Texto,Texto
2,published_at,str,4810,0,0.00,4,Fecha de publicación,Temporal,Fecha y hora
3,updated_at,str,4810,0,0.00,8,Fecha de actualización,Temporal,Fecha y hora
4,category,str,4810,0,0.00,638,Categoría del premio,Categórica,Categoría
5,nominee,str,4804,6,0.12,4131,Nominado,Texto / entidad,Texto
6,artist,str,2970,1840,38.25,1658,Artista asociado,Texto / entidad,Texto
7,workers,str,2620,2190,45.53,2366,Profesionales asociados,Texto / multivaluado,Texto
8,img,str,3443,1367,28.42,1463,Imagen asociada,Texto / URL,URL
9,winner,str,4810,0,0.00,1,Indicador de ganador,Categórica binaria,VERDADERO / FALSO


# 3. Análisis de completitud

En esta etapa se analizará la presencia de valores faltantes en cada atributo del dataset.

El objetivo es determinar:

- Cantidad y porcentaje de valores nulos.
- Presencia de cadenas vacías o valores equivalentes a faltantes.
- Concentración de faltantes por año.
- Concentración de faltantes por categoría.
- Posibles diferencias de completitud entre campos obligatorios y opcionales.

Los valores faltantes no serán tratados ni imputados durante el profiling. Primero se analizará su distribución y posible significado dentro del dominio.

Un porcentaje elevado de valores faltantes no implica necesariamente un problema de calidad. La interpretación dependerá de la naturaleza del atributo y de la granularidad de los registros.

In [92]:
missing_profile = pd.DataFrame({
    "column": df.columns,
    "null_count": df.isna().sum().values,
    "null_percentage": (
        df.isna().mean().values * 100
    ).round(2),
    "empty_string_count": (
        df.fillna("").astype(str).apply(
            lambda col: col.str.strip().eq("").sum()
        ).values
    )
})

missing_profile

,column,null_count,null_percentage,empty_string_count
0,year,0,0.00,0
1,title,0,0.00,0
2,published_at,0,0.00,0
3,updated_at,0,0.00,0
4,category,0,0.00,0
5,nominee,6,0.12,6
6,artist,1840,38.25,1840
7,workers,2190,45.53,2190
8,img,1367,28.42,1367
9,winner,0,0.00,0


In [93]:
missing_by_year = (
    df.groupby("year")
      .agg(
          records=("year", "size"),
          nominee_null=("nominee", lambda x: x.isna().sum()),
          artist_null=("artist", lambda x: x.isna().sum()),
          workers_null=("workers", lambda x: x.isna().sum()),
          img_null=("img", lambda x: x.isna().sum())
      )
      .reset_index()
)

missing_by_year

,year,records,nominee_null,artist_null,workers_null,img_null
0,1958,28,0,13,16,14
1,1959,35,0,17,20,22
2,1960,39,0,18,22,20
3,1961,41,0,22,19,22
4,1962,39,0,21,18,13
...,...,...,...,...,...,...
57,2015,84,0,33,18,14
58,2016,85,0,34,16,13
59,2017,86,0,32,16,19
60,2018,86,0,34,17,17


In [94]:
missing_by_year_pct = missing_by_year.copy()

for column in [
    "nominee_null",
    "artist_null",
    "workers_null",
    "img_null"
]:
    missing_by_year_pct[column] = (
        missing_by_year_pct[column]
        / missing_by_year_pct["records"]
        * 100
    ).round(2)

missing_by_year_pct

,year,records,nominee_null,artist_null,workers_null,img_null
0,1958,28,0.0,46.43,57.14,50.00
1,1959,35,0.0,48.57,57.14,62.86
2,1960,39,0.0,46.15,56.41,51.28
3,1961,41,0.0,53.66,46.34,53.66
4,1962,39,0.0,53.85,46.15,33.33
...,...,...,...,...,...,...
57,2015,84,0.0,39.29,21.43,16.67
58,2016,85,0.0,40.00,18.82,15.29
59,2017,86,0.0,37.21,18.60,22.09
60,2018,86,0.0,39.53,19.77,19.77


In [95]:
missing_by_category = (
    df.groupby("category")
      .agg(
          records=("category", "size"),
          artist_null=("artist", lambda x: x.isna().sum()),
          workers_null=("workers", lambda x: x.isna().sum()),
          img_null=("img", lambda x: x.isna().sum())
      )
      .reset_index()
)

missing_by_category["artist_null_pct"] = (
    missing_by_category["artist_null"]
    / missing_by_category["records"]
    * 100
).round(2)

missing_by_category["workers_null_pct"] = (
    missing_by_category["workers_null"]
    / missing_by_category["records"]
    * 100
).round(2)

missing_by_category["img_null_pct"] = (
    missing_by_category["img_null"]
    / missing_by_category["records"]
    * 100
).round(2)

missing_by_category.sort_values(
    "records",
    ascending=False
).head(20)

,category,records,artist_null,workers_null,img_null,artist_null_pct,workers_null_pct,img_null_pct
636,Song Of The Year,70,70,0,17,100.00,0.00,24.29
632,Record Of The Year,69,0,7,5,0.00,10.14,7.25
2,Album Of The Year,66,0,4,2,0.00,6.06,3.03
409,Best Opera Recording,64,64,0,13,100.00,0.00,20.31
19,Best Album Notes,63,63,0,49,100.00,0.00,77.78
292,Best Instrumental Composition,55,55,0,13,100.00,0.00,23.64
204,Best Country Song,55,55,0,13,100.00,0.00,23.64
395,Best New Artist,51,50,51,44,98.04,100.00,86.27
277,Best Historical Album,44,44,0,32,100.00,0.00,72.73
59,Best Chamber Music Performance,43,43,0,6,100.00,0.00,13.95


# 4. Análisis de unicidad y duplicación

En esta etapa se analizará la unicidad de los registros y de los principales atributos identificadores del dataset.

El objetivo es determinar:

- Si existen filas completamente duplicadas.
- Qué atributos presentan mayor y menor cardinalidad.
- Qué campos podrían participar en la identificación de una nominación.
- Si existen combinaciones de atributos repetidas.
- Si las repeticiones corresponden a duplicación técnica o a múltiples observaciones legítimas.

Se analizarán inicialmente atributos como `year`, `title`, `category`, `nominee` y `artist`, así como combinaciones relacionadas con la posible granularidad del dataset.

No se eliminarán registros durante esta etapa.

In [96]:
full_duplicates = df.duplicated().sum()

print(f"Full duplicate rows: {full_duplicates}")

Full duplicate rows: 0


In [97]:
uniqueness_profile = pd.DataFrame({
    "column": df.columns,
    "total_records": len(df),
    "unique_values": [
        df[column].nunique(dropna=True)
        for column in df.columns
    ]
})

uniqueness_profile["uniqueness_percentage"] = (
    uniqueness_profile["unique_values"]
    / uniqueness_profile["total_records"]
    * 100
).round(2)

uniqueness_profile.sort_values(
    "uniqueness_percentage",
    ascending=False
)

,column,total_records,unique_values,uniqueness_percentage
5,nominee,4810,4131,85.88
7,workers,4810,2366,49.19
6,artist,4810,1658,34.47
8,img,4810,1463,30.42
4,category,4810,638,13.26
0,year,4810,62,1.29
1,title,4810,62,1.29
3,updated_at,4810,8,0.17
2,published_at,4810,4,0.08
9,winner,4810,1,0.02


In [98]:
repeated_nomination_groups = (
    df.groupby(
        ["year", "category", "nominee"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

repeated_nomination_groups = (
    repeated_nomination_groups[
        repeated_nomination_groups["records"] > 1
    ]
    .sort_values("records", ascending=False)
)

repeated_nomination_groups.head(20)

,year,category,nominee,records


# 5. Análisis de variables categóricas

En esta etapa se analizarán las variables categóricas y textuales relevantes para identificar:

- Cardinalidad y distribución de valores.
- Categorías dominantes y poco frecuentes.
- Valores inesperados o inconsistentes.
- Variaciones de escritura, espacios y capitalización.
- Relación entre `year` y `title`.
- Dominio real de `winner`.
- Distribución de categorías por año.

El objetivo es distinguir entre variaciones legítimas propias del dominio de los GRAMMY y problemas de estandarización que deberán ser considerados durante la transformación.

In [99]:
categorical_columns = [
    "year",
    "title",
    "category",
    "winner"
]

categorical_inventory = []

for column in categorical_columns:
    categorical_inventory.append({
        "column": column,
        "unique_values": df[column].nunique(dropna=True),
        "null_values": df[column].isna().sum()
    })

categorical_inventory = pd.DataFrame(categorical_inventory)

categorical_inventory

,column,unique_values,null_values
0,year,62,0
1,title,62,0
2,category,638,0
3,winner,1,0


In [100]:
winner_distribution = (
    df["winner"]
    .value_counts(dropna=False)
    .reset_index()
)

winner_distribution.columns = [
    "winner",
    "records"
]

winner_distribution["percentage"] = (
    winner_distribution["records"]
    / len(df)
    * 100
).round(2)

winner_distribution

,winner,records,percentage
0,VERDADERO,4810,100.0


In [101]:
category_distribution = (
    df["category"]
    .value_counts(dropna=False)
    .reset_index()
)

category_distribution.columns = [
    "category",
    "records"
]

category_distribution["percentage"] = (
    category_distribution["records"]
    / len(df)
    * 100
).round(2)

category_distribution.head(20)

,category,records,percentage
0,Song Of The Year,70,1.46
1,Record Of The Year,69,1.43
2,Album Of The Year,66,1.37
3,Best Opera Recording,64,1.33
4,Best Album Notes,63,1.31
5,Best Country Song,55,1.14
6,Best Instrumental Composition,55,1.14
7,Best New Artist,51,1.06
8,Best Historical Album,44,0.91
9,Best Chamber Music Performance,43,0.89


In [102]:
rare_categories = (
    category_distribution[
        category_distribution["records"] <= 2
    ]
    .sort_values("records")
)

rare_categories

,category,records,percentage
420,Best Traditional Tropical Latin Performance,1,0.02
621,Best Sound Track Album - Background Score From...,1,0.02
622,"Best Sound Track Album, Original Cast - Motion...",1,0.02
623,Best Comedy Performance - Musical,1,0.02
624,"Best Performance By A ""Top 40"" Artist",1,0.02
...,...,...,...
377,Best Album Notes - Classical,2,0.04
378,Best Score From The Original Cast Show Album,2,0.04
348,Best Hard Rock/Metal Performance,2,0.04
365,"Best Album Cover, Photography",2,0.04


In [103]:
year_title_relationship = (
    df[["year", "title"]]
    .drop_duplicates()
    .sort_values(["year", "title"])
)

year_title_relationship

,year,title
4782,1958,1st Annual GRAMMY Awards (1958)
4747,1959,2nd Annual GRAMMY Awards (1959)
4708,1960,3rd Annual GRAMMY Awards (1960)
4160,1961,4th Annual GRAMMY Awards (1961)
4630,1962,5th Annual GRAMMY Awards (1962)
...,...,...
568,2015,58th Annual GRAMMY Awards (2015)
608,2016,59th Annual GRAMMY Awards (2016)
369,2017,60th Annual GRAMMY Awards (2017)
435,2018,61st Annual GRAMMY Awards (2018)


### Consistencia textual básica

Se revisarán espacios externos y patrones de espacios internos en los principales atributos textuales.

El objetivo es identificar inconsistencias de representación que puedan afectar posteriormente la estandarización, comparación o agrupación de valores.

No se realizarán modificaciones sobre los datos RAW.

In [104]:
text_columns = [
    "title",
    "category",
    "nominee",
    "artist",
    "workers"
]

text_consistency = []

for column in text_columns:
    values = df[column].dropna().astype(str)

    leading_spaces = values.str.match(r"^\s").sum()
    trailing_spaces = values.str.match(r".*\s$").sum()
    multiple_spaces = values.str.contains(r"\s{2,}", regex=True).sum()

    text_consistency.append({
        "column": column,
        "leading_spaces": leading_spaces,
        "trailing_spaces": trailing_spaces,
        "multiple_spaces": multiple_spaces
    })

text_consistency = pd.DataFrame(text_consistency)

text_consistency

,column,leading_spaces,trailing_spaces,multiple_spaces
0,title,0,0,4810
1,category,0,0,2
2,nominee,0,0,7
3,artist,0,0,1
4,workers,0,0,9


# 6. Análisis temporal

En esta etapa se analizarán las variables temporales del dataset:

- `year`
- `published_at`
- `updated_at`

El objetivo es determinar:

- Rango temporal cubierto por el dataset.
- Distribución de registros por año.
- Formato y capacidad de conversión de las fechas.
- Relación entre las fechas de publicación y actualización.
- Diferencia temporal entre `published_at` y `updated_at`.
- Relación entre `year` y las fechas de publicación/actualización.

Es importante distinguir entre `year`, que representa el año de la edición de los premios, y `published_at` / `updated_at`, que representan fechas asociadas a la publicación y actualización de los registros en la fuente.

Durante esta etapa las fechas se convertirán únicamente en columnas auxiliares para el análisis. No se modificarán los datos almacenados en RAW.

In [105]:
temporal_columns = [
    "year",
    "published_at",
    "updated_at"
]

for column in temporal_columns:
    print(f"\n--- {column} ---")
    print(f"Unique values: {df[column].nunique(dropna=True)}")
    print(df[column].dropna().unique()[:20])


--- year ---
Unique values: 62
<StringArray>
['2019', '1986', '2017', '2018', '2015', '2014', '2016', '1975', '2013',
 '1985', '2012', '2011', '2010', '1974', '1984', '2009', '2008', '2007',
 '2006', '1983']
Length: 20, dtype: str

--- published_at ---
Unique values: 4
<StringArray>
['19/05/2020 5:10', '28/11/2017 0:03', '22/05/2018 3:08', '6/12/2018 23:48']
Length: 4, dtype: str

--- updated_at ---
Unique values: 8
<StringArray>
['19/05/2020 5:10', '10/09/2019 1:06', '10/09/2019 1:09', '1/09/2020 12:16',
 '10/09/2019 1:07', '10/09/2019 1:08', '10/09/2019 1:11', '28/11/2017 0:03']
Length: 8, dtype: str


In [106]:
df_temporal = df.copy()

df_temporal["year_num"] = pd.to_numeric(
    df_temporal["year"],
    errors="coerce"
)

df_temporal["published_at_dt"] = pd.to_datetime(
    df_temporal["published_at"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

df_temporal["updated_at_dt"] = pd.to_datetime(
    df_temporal["updated_at"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

temporal_parse = pd.DataFrame({
    "column": [
        "year",
        "published_at",
        "updated_at"
    ],
    "original_nulls": [
        df["year"].isna().sum(),
        df["published_at"].isna().sum(),
        df["updated_at"].isna().sum()
    ],
    "parse_failures": [
        df_temporal["year_num"].isna().sum(),
        df_temporal["published_at_dt"].isna().sum(),
        df_temporal["updated_at_dt"].isna().sum()
    ]
})

temporal_parse

,column,original_nulls,parse_failures
0,year,0,0
1,published_at,0,0
2,updated_at,0,0


In [107]:
year_distribution = (
    df_temporal["year_num"]
    .value_counts()
    .sort_index()
    .reset_index()
)

year_distribution.columns = [
    "year",
    "records"
]

year_distribution

,year,records
0,1958,28
1,1959,35
2,1960,39
3,1961,41
4,1962,39
...,...,...
57,2015,84
58,2016,85
59,2017,86
60,2018,86


In [108]:
print(f"Minimum year: {df_temporal['year_num'].min():.0f}")
print(f"Maximum year: {df_temporal['year_num'].max():.0f}")
print(f"Number of years: {df_temporal['year_num'].nunique()}")

Minimum year: 1958
Maximum year: 2019
Number of years: 62


In [109]:
temporal_consistency = pd.DataFrame({
    "published_before_updated": (
        df_temporal["updated_at_dt"]
        >= df_temporal["published_at_dt"]
    )
})

temporal_consistency["published_at"] = df_temporal["published_at_dt"]
temporal_consistency["updated_at"] = df_temporal["updated_at_dt"]

temporal_consistency["update_delay"] = (
    temporal_consistency["updated_at"]
    - temporal_consistency["published_at"]
)

temporal_consistency["published_before_updated"].value_counts()

published_before_updated
True    4810
Name: count, dtype: int64

In [110]:
update_delay_profile = (
    temporal_consistency["update_delay"]
    .value_counts()
    .reset_index()
)

update_delay_profile.columns = [
    "update_delay",
    "records"
]

update_delay_profile.head(20)

,update_delay,records
0,651 days 01:03:00,1263
1,651 days 01:05:00,778
2,651 days 01:04:00,713
3,651 days 01:08:00,706
4,651 days 01:06:00,554
5,0 days 00:00:00,541
6,475 days 21:58:00,86
7,277 days 01:18:00,86
8,1008 days 12:13:00,83


In [111]:
year_temporal_profile = (
    df_temporal
    .groupby("year_num")
    .agg(
        records=("year_num", "size"),
        published_dates=("published_at_dt", "nunique"),
        updated_dates=("updated_at_dt", "nunique")
    )
    .reset_index()
)

year_temporal_profile

,year_num,records,published_dates,updated_dates
0,1958,28,1,1
1,1959,35,1,1
2,1960,39,1,1
3,1961,41,1,1
4,1962,39,1,1
...,...,...,...,...
57,2015,84,1,1
58,2016,85,1,1
59,2017,86,1,1
60,2018,86,1,1


In [112]:
year_timestamp_relationship = (
    df_temporal[
        [
            "year_num",
            "published_at_dt",
            "updated_at_dt"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "year_num",
            "published_at_dt",
            "updated_at_dt"
        ]
    )
)

year_timestamp_relationship

,year_num,published_at_dt,updated_at_dt
4782,1958,2017-11-28 00:03:00,2019-09-10 01:11:00
4747,1959,2017-11-28 00:03:00,2019-09-10 01:06:00
4708,1960,2017-11-28 00:03:00,2019-09-10 01:07:00
4160,1961,2017-11-28 00:03:00,2019-09-10 01:08:00
4630,1962,2017-11-28 00:03:00,2019-09-10 01:09:00
...,...,...,...
568,2015,2017-11-28 00:03:00,2019-09-10 01:09:00
608,2016,2017-11-28 00:03:00,2019-09-10 01:06:00
369,2017,2018-05-22 03:08:00,2019-09-10 01:06:00
435,2018,2018-12-06 23:48:00,2019-09-10 01:06:00


# 7. Análisis de granularidad y relaciones

En esta etapa se determinará la granularidad lógica del dataset y se evaluarán las combinaciones de atributos que podrían identificar de manera única una observación.

El objetivo es establecer:

- Qué representa una fila dentro del dataset.
- Si `year + category + nominee` identifica una nominación de manera única.
- Si existen múltiples artistas o profesionales asociados a una misma nominación.
- Si `artist` y `workers` modifican la granularidad o representan atributos complementarios.
- Qué combinación de atributos puede utilizarse posteriormente como clave de negocio.

No se realizarán modificaciones sobre los datos RAW.

In [113]:
grain_key = [
    "year",
    "category",
    "nominee"
]

grain_profile = (
    df.groupby(
        grain_key,
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

grain_profile_summary = pd.DataFrame({
    "total_records": [len(df)],
    "unique_combinations": [len(grain_profile)],
    "repeated_combinations": [
        (grain_profile["records"] > 1).sum()
    ],
    "maximum_records_per_combination": [
        grain_profile["records"].max()
    ]
})

grain_profile_summary

,total_records,unique_combinations,repeated_combinations,maximum_records_per_combination
0,4810,4810,0,1


In [114]:
artist_multiplicity = (
    df["artist"]
    .dropna()
    .astype(str)
    .str.split(",")
    .str.len()
    .value_counts()
    .sort_index()
    .reset_index()
)

artist_multiplicity.columns = [
    "number_of_values",
    "records"
]

artist_multiplicity

,number_of_values,records
0,1,2846
1,2,78
2,3,18
3,4,11
4,5,5
5,6,4
6,7,3
7,8,2
8,10,1
9,12,2


In [115]:
workers_multiplicity = (
    df["workers"]
    .dropna()
    .astype(str)
    .str.split(",")
    .str.len()
    .value_counts()
    .sort_index()
    .reset_index()
)

workers_multiplicity.columns = [
    "number_of_values",
    "records"
]

workers_multiplicity

,number_of_values,records
0,1,110
1,2,1142
2,3,678
3,4,342
4,5,132
5,6,66
6,7,53
7,8,35
8,9,24
9,10,12


In [116]:
print("ARTIST — ejemplos multivaluados:\n")

artist_examples = (
    df[
        df["artist"].notna()
        & df["artist"].astype(str).str.contains(",")
    ][["year", "category", "nominee", "artist"]]
    .head(10)
)

artist_examples

ARTIST — ejemplos multivaluados:



,year,category,nominee,artist
57,2019,Best Dance Recording,Midnight Hour,"Skrillex, Boys Noize & Ty Dolla $ign"
122,2019,Best Rap Performance,Down Bad,"Dreamville Featuring J.I.D, Bas, J. Cole, EART..."
134,2019,Best Rap Album,Igor,"Tyler, The Creator"
186,2019,Best Latin Jazz Album,Sorte!: Music By John Finbury,"Thalma de Freitas With Vitor Gonçalves, John P..."
457,2018,Best Rap Performance,King's Dead,"Kendrick Lamar, Jay Rock, Future & James Blake"
583,2017,Best Song Written For Visual Media,How Far I'll Go,"Lin-Manuel Miranda, songwriter (Auli'i Cravalho)"
668,2016,Best Song Written For Visual Media,Can't Stop The Feeling!,"Max Martin, Shellback & Justin Timberlake, son..."
715,2015,Best Rap/Sung Collaboration,These Walls,"Kendrick Lamar Featuring Bilal, Anna Wise & Th..."
754,2015,Best Song Written For Visual Media,Glory,"Lonnie Lynn, Che Smith & John Stephens, songwr..."
837,2014,Best Song Written For Visual Media,Let It Go,"Kristen Anderson-Lopez & Robert Lopez, songwri..."


In [117]:
print("WORKERS — ejemplos multivaluados:\n")

workers_examples = (
    df[
        df["workers"].notna()
        & df["workers"].astype(str).str.contains(",")
    ][["year", "category", "nominee", "workers"]]
    .head(10)
    )
workers_examples

WORKERS — ejemplos multivaluados:



,year,category,nominee,workers
0,2019,Record Of The Year,Bad Guy,"Finneas O'Connell, producer; Rob Kinelski & Fi..."
1,2019,Record Of The Year,"Hey, Ma","BJ Burton, Brad Cook, Chris Messina & Justin V..."
2,2019,Record Of The Year,7 rings,"Charles Anderson, Tommy Brown, Michael Foster ..."
3,2019,Record Of The Year,Hard Place,"Rodney “Darkchild” Jerkins, producer; Joseph H..."
4,2019,Record Of The Year,Talk,"Disclosure & Denis Kosiak, producers; Ingmar C..."
5,2019,Record Of The Year,Old Town Road,"Andrew ""VoxGod"" Bolooki, Jocelyn “Jozzy” Donal..."
6,2019,Record Of The Year,Truth Hurts,"Ricky Reed & Tele, producers; Chris Galland, M..."
7,2019,Record Of The Year,Sunflower,"Louis Bell & Carter Lang, producers; Louis Bel..."
8,2019,Album Of The Year,"When We All Fall Asleep, Where Do We Go?","Finneas O'Connell, producer; Rob Kinelski & Fi..."
9,2019,Album Of The Year,"I,I","Brad Cook, Chris Messina & Justin Vernon, prod..."


# 8. Análisis de calidad semántica

En esta etapa se evaluará la coherencia de los valores respecto a su significado de negocio.

El objetivo es identificar:

- Valores inesperados en variables categóricas.
- Consistencia entre `year` y `title`.
- Valores faltantes en atributos que podrían ser necesarios para identificar una nominación.
- Posibles inconsistencias semánticas que deberán considerarse durante la transformación.

Este análisis busca diferenciar problemas reales de calidad de datos de características legítimas de la fuente.

In [118]:
semantic_domains = {
    "winner": df["winner"].dropna().unique(),
    "year": sorted(
        pd.to_numeric(
            df["year"],
            errors="coerce"
        ).dropna().unique()
    )
}

for column, values in semantic_domains.items():
    print(f"\n--- {column} ---")
    print(values)

print("\nMissing values in grain attributes:")

print(
    df[
        ["year", "category", "nominee"]
    ].isna().sum()
)


--- winner ---
<StringArray>
['VERDADERO']
Length: 1, dtype: str

--- year ---
[np.int64(1958), np.int64(1959), np.int64(1960), np.int64(1961), np.int64(1962), np.int64(1963), np.int64(1964), np.int64(1965), np.int64(1966), np.int64(1967), np.int64(1968), np.int64(1969), np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1974), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int6

In [119]:
missing_nominee = (
    df[
        df["nominee"].isna()
    ][
        [
            "year",
            "title",
            "category",
            "nominee",
            "artist",
            "workers",
            "winner"
        ]
    ]
)

missing_nominee

,year,title,category,nominee,artist,workers,winner
2073,1997,40th Annual GRAMMY Awards (1997),"Remixer Of The Year, Non-Classical",NaN,NaN,NaN,VERDADERO
2278,2000,43rd Annual GRAMMY Awards (2000),"Remixer of the Year, Non-Classical",NaN,NaN,NaN,VERDADERO
2377,1999,42nd Annual GRAMMY Awards (1999),"Remixer Of The Year, Non-Classical",NaN,NaN,NaN,VERDADERO
2472,1998,41st Annual GRAMMY Awards (1998),"Remixer Of The Year, Non-classical",NaN,NaN,NaN,VERDADERO
4531,1965,8th Annual GRAMMY Awards (1965),Best New Country & Western Artist,NaN,NaN,NaN,VERDADERO
4576,1964,7th Annual GRAMMY Awards (1964),Best New Country & Western Artist Of 1964,NaN,NaN,NaN,VERDADERO


In [120]:
category_normalization = (
    df["category"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

category_comparison = pd.DataFrame({
    "original_unique": [df["category"].nunique()],
    "normalized_unique": [category_normalization.nunique()]
})

category_comparison

,original_unique,normalized_unique
0,638,630


In [121]:
category_variants = (
    pd.DataFrame({
        "original": df["category"].dropna().unique()
    })
)

category_variants["normalized"] = (
    category_variants["original"]
    .str.strip()
    .str.lower()
)

category_variants = (
    category_variants
    .groupby("normalized")["original"]
    .agg(list)
    .reset_index()
)

category_variants[
    category_variants["original"].str.len() > 1
].head(30)

,normalized,original
96,best classical performance - instrumental solo...,[Best Classical Performance - Instrumental Sol...
307,best instrumental soloist performance (without...,[Best Instrumental Soloist Performance (withou...
310,best instrumental soloist(s) performance (with...,[Best Instrumental Soloist(s) Performance (wit...
536,best small ensemble performance (with or witho...,[Best Small Ensemble Performance (with or with...
567,"best southern, country, or bluegrass gospel album","[Best Southern, Country, Or Bluegrass Gospel A..."
572,best spoken word album for children,"[Best Spoken Word Album For Children, Best Spo..."
627,"remixer of the year, non-classical","[Remixer Of The Year, Non-Classical, Remixer o..."


In [122]:
category_variants_full = (
    df["category"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

category_variants_full = (
    pd.DataFrame({
        "original": df["category"].dropna().astype(str).unique()
    })
)

category_variants_full["normalized"] = (
    category_variants_full["original"]
    .str.strip()
    .str.lower()
)

category_variants_full = (
    category_variants_full
    .groupby("normalized")["original"]
    .agg(list)
    .reset_index()
)

category_variants_full = category_variants_full[
    category_variants_full["original"].str.len() > 1
]

for _, row in category_variants_full.iterrows():
    print("\nNORMALIZED:")
    print(row["normalized"])
    print("VARIANTS:")
    for variant in row["original"]:
        print(f" - {variant}")


NORMALIZED:
best classical performance - instrumental soloist or soloists (with orchestra)
VARIANTS:
 - Best Classical Performance - Instrumental Soloist Or Soloists (With Orchestra)
 - Best Classical Performance - Instrumental Soloist OR Soloists (With Orchestra)

NORMALIZED:
best instrumental soloist performance (without orchestra)
VARIANTS:
 - Best Instrumental Soloist Performance (without Orchestra)
 - Best Instrumental Soloist Performance (Without Orchestra)

NORMALIZED:
best instrumental soloist(s) performance (with orchestra)
VARIANTS:
 - Best Instrumental Soloist(s) Performance (with Orchestra)
 - Best Instrumental Soloist(s) Performance (With Orchestra)

NORMALIZED:
best small ensemble performance (with or without conductor)
VARIANTS:
 - Best Small Ensemble Performance (with or without Conductor)
 - Best Small Ensemble Performance (With Or Without Conductor)

NORMALIZED:
best southern, country, or bluegrass gospel album
VARIANTS:
 - Best Southern, Country, Or Bluegrass Gospel

In [123]:
title_normalization = pd.DataFrame({
    "original": df["title"].dropna().unique()
})

title_normalization["normalized"] = (
    title_normalization["original"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Original unique titles:", title_normalization["original"].nunique())
print("Normalized unique titles:", title_normalization["normalized"].nunique())

title_variants = (
    title_normalization
    .groupby("normalized")["original"]
    .agg(list)
    .reset_index()
)

title_variants[
    title_variants["original"].str.len() > 1
]

Original unique titles: 62
Normalized unique titles: 62


,normalized,original


# 9. Consolidación del perfilamiento

El perfilamiento realizado sobre `raw.grammy` permitió identificar las principales características estructurales, de completitud, unicidad, temporalidad, granularidad y calidad semántica del dataset.

Los hallazgos obtenidos se resumen a continuación.

## 9.1 Perfil general

- El dataset contiene **4,810 registros y 10 columnas**.
- No existen filas completamente duplicadas.
- `year` contiene **62 valores únicos**, correspondientes al periodo **1958–2019**.
- `title` contiene **62 valores únicos** y mantiene una relación 1:1 con `year`.
- `category` contiene **638 valores únicos**.
- `nominee` contiene **4,131 valores únicos**.
- `winner` presenta un único valor en todo el dataset, por lo que es un atributo constante.

## 9.2 Completitud

Los principales valores faltantes identificados fueron:

| Campo | Nulos | Porcentaje | Interpretación |
|---|---:|---:|---|
| `nominee` | 6 | 0.12% | Casos puntuales que requieren tratamiento especial |
| `artist` | 1,840 | 38.25% | Ausencia asociada a determinadas categorías |
| `workers` | 2,190 | 45.53% | Ausencia asociada a determinadas categorías |
| `img` | 1,367 | 28.42% | Campo auxiliar de la fuente |

El análisis por categoría mostró que los valores faltantes de `artist`, `workers` e `img` no se distribuyen de manera aleatoria. Existen categorías donde estos atributos no aplican o no fueron registrados por la fuente.

Por esta razón:

- No se realizará imputación general de `artist`, `workers` ni `img`.
- La ausencia de estos campos no será considerada automáticamente como un error crítico.
- `nominee` requiere un tratamiento diferenciado debido a que únicamente existen 6 registros sin valor.

## 9.3 Unicidad y granularidad

No se identificaron duplicados completos.

La combinación:

`year + category + nominee`

identifica de manera única las **4,810 observaciones** cuando `nominee` está presente.

Por tanto, la granularidad lógica del dataset se establece como:

> **Una observación representa una nominación dentro de una categoría de los premios GRAMMY para una determinada edición/año.**

Sin embargo, existen 6 registros con `nominee` nulo. Estos registros corresponden a categorías específicas y no deben ser eliminados ni rellenados artificialmente durante el profiling.

`artist` y `workers` pueden contener múltiples personas o profesionales dentro de una misma celda. Esto no modifica la granularidad del registro, por lo que no se dividirán automáticamente durante la transformación sin una justificación analítica adicional.

## 9.4 Variables categóricas

`category` presenta **638 valores originales**.

Al normalizar espacios y capitalización se obtuvieron **630 valores**, identificándose 8 agrupaciones con diferencias únicamente de representación textual.

Ejemplos:

- Diferencias entre `Or` y `OR`.
- Diferencias entre `With` y `with`.
- Diferencias entre `Without` y `without`.
- Diferencias de capitalización en `Remixer Of The Year` / `Remixer of the Year`.

Estas diferencias no representan, según el análisis realizado, categorías de negocio distintas.

**Decisión:** durante la transformación se podrá estandarizar la representación textual de `category`, manteniendo RAW sin modificaciones.

## 9.5 Variables temporales

`year` cubre el periodo **1958–2019**, sin saltos en la secuencia de años.

Los campos `published_at` y `updated_at`:

- Presentan formatos convertibles correctamente.
- No contienen errores de parsing.
- Cumplen `updated_at >= published_at` en los 4,810 registros.
- Presentan pocos valores distintos y se repiten entre múltiples registros.

Por su comportamiento, se interpretan como **metadatos de publicación y actualización de la fuente**, no como fechas del evento GRAMMY.

**Decisión:**

- `year` será utilizado como referencia temporal de negocio.
- `published_at` y `updated_at` se conservarán como metadatos de trazabilidad.

## 9.6 Calidad semántica

Se identificaron 6 registros con `nominee` nulo. Estos corresponden principalmente a categorías históricas específicas, por lo que no se consideran evidencia suficiente para imputar un valor.

Asimismo, `winner` contiene un único valor en todos los registros. Esto indica que el dataset representa registros de ganadores y no proporciona variabilidad suficiente para utilizar este atributo como indicador analítico.

## 9.7 Riesgos identificados

| ID | Riesgo | Evidencia | Impacto |
|---|---|---|---|
| R-GRM-01 | Valores faltantes en atributos descriptivos | `artist`, `workers`, `img` presentan porcentajes elevados de nulos | Medio |
| R-GRM-02 | `nominee` ausente en algunos registros | 6 registros sin `nominee` | Alto para identificación |
| R-GRM-03 | Variaciones de capitalización en categorías | 638 → 630 categorías normalizadas | Medio |
| R-GRM-04 | Atributos multivaluados | `artist` y `workers` contienen múltiples entidades | Medio |
| R-GRM-05 | `published_at` y `updated_at` no representan el año del premio | Timestamps compartidos entre múltiples registros | Bajo |
| R-GRM-06 | `winner` es constante | Un único valor en 4,810 registros | Bajo |

## 9.8 Decisiones preliminares de transformación

A partir del profiling se establecen las siguientes decisiones para la etapa de transformación:

1. Mantener la granularidad de una fila por nominación.
2. No eliminar duplicados completos, ya que no fueron encontrados.
3. No imputar de forma general `artist`, `workers` ni `img`.
4. Tratar los 6 registros con `nominee` nulo mediante una regla específica, sin inventar valores.
5. Estandarizar la representación textual de `category`.
6. Evaluar la normalización de `artist` y `workers` sin fragmentar automáticamente su contenido.
7. Convertir `year` al tipo temporal apropiado para el modelo dimensional.
8. Convertir `published_at` y `updated_at` a tipos temporales conservando su función de trazabilidad.
9. Mantener `winner` por fidelidad de la fuente, aunque su utilidad analítica sea limitada.

## 9.9 Conclusión del profiling

El dataset presenta una estructura consistente y no contiene duplicados completos ni errores temporales evidentes. Los principales desafíos de calidad se concentran en los valores faltantes dependientes de la categoría, la ausencia puntual de `nominee`, la representación multivaluada de `artist` y `workers` y las variaciones de capitalización en `category`.

Estos hallazgos servirán como fundamento para definir las reglas de calidad de datos y las transformaciones posteriores. Las reglas de validación deberán distinguir entre **errores críticos**, **condiciones aceptables de la fuente** y **atributos opcionales**, evitando convertir características legítimas del dominio en falsos errores de calidad.